# Pre-Pilot Sanity Check

Quick analysis of a batch run output. Load the JSONL sessions, check per-cell unsafe rates, and verify the pipeline is producing sensible data before committing to a full 840-cell experiment.

In [ ]:
from pathlib import Path
import pandas as pd
import json

In [ ]:
# Load sessions from JSONL
SESSION_PATH = Path('../data/sessions/pre_pilot_001.jsonl')

if not SESSION_PATH.exists():
    # Fall back to mock output if real pilot hasn't run yet
    SESSION_PATH = Path('../data/sessions/pre_pilot_mock.jsonl')

sessions_raw = []
with open(SESSION_PATH) as f:
    for line in f:
        if line.strip():
            sessions_raw.append(json.loads(line))

print(f'Loaded {len(sessions_raw)} sessions from {SESSION_PATH.name}')

In [ ]:
# Flatten into a DataFrame
rows = []
for s in sessions_raw:
    row = {
        'session_id': s['session_id'],
        'scenario_id': s['cell']['scenario_id'],
        'pressure_id': s['cell']['pressure_id'],
        'monitoring_id': s['cell']['monitoring_id'],
        'model_name': s['cell']['model_name'],
        'seed': s['cell']['seed'],
        'overall_unsafe': s['layer_a']['overall_unsafe'],
        'n_triggered': len(s['layer_a']['triggered_signal_ids'] if 'triggered_signal_ids' in s['layer_a'] else [sig for sig in s['layer_a']['signals'] if s['layer_a']['signals'][sig]['triggered']]),
        'duration_s': s['duration_seconds'],
    }
    # Add Layer B scores if present
    if s.get('layer_b'):
        for dim in ['sycophancy', 'coherence', 'hedging_faithfulness', 'evidence_faithfulness', 'overall_alignment']:
            row[f'judge_{dim}'] = s['layer_b']['scores'][dim]
    rows.append(row)

df = pd.DataFrame(rows)
df.head()

In [ ]:
# Per-cell unsafe rate by pressure × monitoring
print('=== Unsafe rate by pressure × monitoring ===')
pivot = df.groupby(['pressure_id', 'monitoring_id'])['overall_unsafe'].mean()
print(pivot)
print()

# Per-scenario unsafe rate
print('=== Unsafe rate by scenario ===')
print(df.groupby('scenario_id')['overall_unsafe'].mean())
print()

# Mean triggered signals
print('=== Mean triggered signals by pressure ===')
print(df.groupby('pressure_id')['n_triggered'].mean())

In [ ]:
# Layer B judge scores (if available)
judge_cols = [c for c in df.columns if c.startswith('judge_')]
if judge_cols:
    print('=== Mean Layer B scores by pressure ===')
    print(df.groupby('pressure_id')[judge_cols].mean())
    print()
    print('=== Mean Layer B scores by monitoring ===')
    print(df.groupby('monitoring_id')[judge_cols].mean())
else:
    print('No Layer B scores available (judge was not run).')

In [ ]:
# Quick sanity checks
print(f'\n=== Sanity Checks ===')
print(f'Total sessions: {len(df)}')
print(f'Unique scenarios: {df["scenario_id"].nunique()}')
print(f'Unique pressures: {df["pressure_id"].nunique()}')
print(f'Unique monitoring: {df["monitoring_id"].nunique()}')
print(f'Unique models: {df["model_name"].nunique()}')
print(f'Unique seeds: {df["seed"].nunique()}')
print(f'Overall unsafe rate: {df["overall_unsafe"].mean():.2%}')
print(f'Mean duration: {df["duration_s"].mean():.3f}s')
print()
if df['overall_unsafe'].mean() == 0.0:
    print('WARNING: No unsafe sessions detected. This may indicate the model is well-aligned or the mock adapter is in use.')
elif df['overall_unsafe'].mean() == 1.0:
    print('WARNING: ALL sessions unsafe. Likely using mock adapter (which fails all checks).')